<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# TelcomCI — RH & Masse Salariale
## Notebook 1 — Contexte, Découverte des Données & Premiers KPIs
### ✅ VERSION CORRIGÉE

> **Comment lire ce corrigé :**  
> Les blocs `MÉTHODE` expliquent les choix techniques et les patterns généralisables.  
> Les blocs `INTERPRÉTATION` lisent les résultats.  
> Les blocs `MÉTIER` font le lien entre le chiffre et la décision business.

| | |
|---|---|
| **Entreprise** | TelcomCI — opérateur télécom, Abidjan, Côte d'Ivoire |
| **Période** | Janvier 2021 — Juin 2024 |
| **Niveau** | Avancé |
| **Outils** | Python, DuckDB (JupySQL) |
| **Durée estimée** | 3h à 4h |

---
> TelcomCI est un opérateur télécom ivoirien avec **510 employés** répartis dans **10 départements**. La DRH fait face à trois problèmes urgents :
>
> *"Nous ne savons pas exactement combien nous coûte la masse salariale par département. Notre taux de turnover est trop élevé et nous ne savons pas pourquoi. Les absences non justifiées ont augmenté sans que personne ne l'ait détecté."*

---
## 0. Mise en place de l'environnement

In [1]:
!pip install jupysql==0.11.1 duckdb-engine --quiet


[notice] A new release of pip is available: 24.1 -> 26.0.1
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import duckdb
import os, sys, warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)

COLORS = {
    'primary':   '#534AB7',
    'secondary': '#1D9E75',
    'warning':   '#EF9F27',
    'danger':    '#E24B4A',
    'neutral':   '#888780',
}

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F9F9F8',
    'axes.grid':        True,
    'grid.alpha':       0.35,
    'font.size':        11,
})

# ── Détection Colab / Local ──────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/rh_analytics/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f'📁 Environnement : {"Colab" if IN_COLAB else "Local"}')
print(f'📁 Dossier       : {SAVE_PATH}')
print('Configuration chargée ✅')

📁 Environnement : Local
📁 Dossier       : ./outputs/
Configuration chargée ✅


---
## 1. Connexion DuckDB et chargement des tables

### MÉTHODE — Ordre de chargement (dépendances)

En SQL Server, les tables avec clés étrangères (FK) ne peuvent être créées qu'après les tables qu'elles référencent. Dans DuckDB, les FK ne sont pas enforced mais on conserve l'ordre logique pour la lisibilité :

```
1. departements   (aucune FK)
2. employes        (→ departements)
3. salaires        (→ employes, departements)
4. absences        (→ employes, departements)
5. recrutements    (→ departements)
6. evaluations     (→ employes, departements)
```

> **DuckDB :** `read_csv_auto()` remplace `BULK INSERT`. Il charge le CSV directement depuis GitHub, infère les types automatiquement et crée la table en une seule instruction — zéro configuration.

In [3]:
BASE_URL = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/rh_analytics/data/'

conn = duckdb.connect()
conn.execute(f"""
    CREATE TABLE departements  AS SELECT * FROM read_csv_auto('{BASE_URL}departements.csv');
    CREATE TABLE employes      AS SELECT * FROM read_csv_auto('{BASE_URL}employes.csv',      timestampformat='%Y-%m-%d');
    CREATE TABLE salaires      AS SELECT * FROM read_csv_auto('{BASE_URL}salaires.csv');
    CREATE TABLE absences      AS SELECT * FROM read_csv_auto('{BASE_URL}absences.csv',      timestampformat='%Y-%m-%d');
    CREATE TABLE recrutements  AS SELECT * FROM read_csv_auto('{BASE_URL}recrutements.csv',  timestampformat='%Y-%m-%d');
    CREATE TABLE evaluations   AS SELECT * FROM read_csv_auto('{BASE_URL}evaluations.csv');
""")

result = conn.execute("""
    SELECT 'departements' AS table_name, COUNT(*) AS nb_lignes, 10   AS attendu FROM departements UNION ALL
    SELECT 'employes',                   COUNT(*),              510              FROM employes     UNION ALL
    SELECT 'salaires',                   COUNT(*),              17319            FROM salaires     UNION ALL
    SELECT 'absences',                   COUNT(*),              622              FROM absences     UNION ALL
    SELECT 'recrutements',               COUNT(*),              350              FROM recrutements UNION ALL
    SELECT 'evaluations',                COUNT(*),              1221             FROM evaluations
""").df()
display(result)
print('✅ 6 tables chargées')

,table_name,nb_lignes,attendu
0,departements,10,10
1,employes,510,510
2,salaires,17319,17319
3,absences,622,622
4,recrutements,350,350
5,evaluations,1221,1221


✅ 6 tables chargées


In [4]:
%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
print('%%sql prêt ✅')

%%sql prêt ✅


---
## 2. Exploration des tables

### MÉTHODE — Comprendre le schéma de relations

Avant d'analyser, il faut connaître les clés de jointure entre tables :

```
departements
    ↓ departement_id
employes ──────────────────────────┐
    ↓ employe_id                   ↓ departement_id
salaires    absences    evaluations    recrutements
```

> **DuckDB :** `LIMIT 5` remplace `TOP 5` de SQL Server. La syntaxe `LIMIT` s'écrit en fin de requête, après `ORDER BY`.

In [5]:
%%sql
-- Explorer les 5 premières lignes de chaque table
SELECT * FROM departements LIMIT 5

,departement_id,nom,code,effectif_cible,responsable,budget_annuel_fcfa,localisation
0,DEP01,Direction Generale,DG,5,Laure Diallo,43123780,Abidjan — Plateau
1,DEP02,Commercial & Ventes,COM,95,Kouassi Bamba,1274680740,Abidjan — Plateau
2,DEP03,Technique & Reseau,TECH,110,Luc Toure,1384020000,Abidjan — Plateau
3,DEP04,Service Client,SC,130,Eric Diallo,1439379370,Abidjan — Plateau
4,DEP05,Finance & Comptabilite,FIN,35,Aida Diallo,386088710,Abidjan — Plateau


In [6]:
%%sql
-- Schéma complet des colonnes (équivalent INFORMATION_SCHEMA)
SELECT table_name, column_name, data_type, is_nullable
FROM information_schema.columns
WHERE table_schema = 'main'
ORDER BY table_name, ordinal_position

,table_name,column_name,data_type,is_nullable
0,absences,absence_id,VARCHAR,YES
1,absences,employe_id,VARCHAR,YES
2,absences,departement_id,VARCHAR,YES
3,absences,date_debut,DATE,YES
4,absences,date_fin,DATE,YES
...,...,...,...,...
61,salaires,prime,BIGINT,YES
62,salaires,cotisations,BIGINT,YES
63,salaires,impot_revenu,BIGINT,YES
64,salaires,salaire_net,BIGINT,YES


> **INTERPRÉTATION — Structure des tables :**
>
> - **employes** : 16 colonnes — profil complet avec `date_embauche`, `salaire_base`, `statut` et `date_depart` (NULL pour les actifs)
> - **salaires** : 12 colonnes — un enregistrement par employé × mois, avec décomposition brut/primes/cotisations/net
> - **absences** : 9 colonnes — les champs `justifiee` (booléen) et `impact_salaire` permettront les analyses d'absentéisme
> - **evaluations** : 10 colonnes — 3 notes individuelles + note_globale précalculée + recommandation RH

---
## 3. Diagnostic qualité

### MÉTHODE — Pourquoi diagnostiquer avant d'analyser ?

Analyser des données sans vérifier leur qualité donne des KPIs faux. Exemple : un salaire négatif inclus dans une moyenne salariale la fausse sans qu'aucune erreur SQL ne soit levée. La règle : **toujours diagnostiquer avant d'analyser.**

Le pattern `SUM(CASE WHEN condition THEN 1 ELSE 0 END)` compte les lignes qui satisfont une condition sans filtrer les autres — on voit ainsi les anomalies en proportion du total.

In [23]:
%%sql 
-- DIAGNOSTIC 1 : Valeurs NULL et anomalies dans employes
SELECT
    COUNT(*)                                                      AS total_employes,
    SUM(CASE WHEN email          IS NULL THEN 1 ELSE 0 END)       AS null_email,
    SUM(CASE WHEN salaire_base   IS NULL THEN 1 ELSE 0 END)       AS null_salaire,
    SUM(CASE WHEN departement_id IS NULL THEN 1 ELSE 0 END)       AS null_dept,
    SUM(CASE WHEN salaire_base   <= 0    THEN 1 ELSE 0 END)       AS salaires_aberrants,
    SUM(CASE WHEN salaire_base   <  0    THEN 1 ELSE 0 END)       AS salaires_negatifs,
    SUM(CASE WHEN salaire_base   =  0    THEN 1 ELSE 0 END)       AS salaires_nuls
FROM employes

,total_employes,null_email,null_salaire,null_dept,salaires_aberrants,salaires_negatifs,salaires_nuls
0,510,0.00,0.00,0.00,2.00,1.00,1.00


> **INTERPRÉTATION — Salaires aberrants :**
>
> **2 salaires aberrants** identifiés dans la table `employes` :
> - 1 salaire **négatif** (`-50 000 FCFA`) — saisie erronée probable
> - 1 salaire **nul** (`0 FCFA`) — employé sans contrat actif ou donnée manquante
>
> Ces 2 enregistrements seront **exclus** dans la vue `vw_employes_propres` du NB2 (`WHERE salaire_base > 0`). On ne les supprime pas de la table source pour conserver la traçabilité des données brutes.

In [24]:
%%sql 
-- DIAGNOSTIC 2 : Emails dupliqués
SELECT
    email,
    COUNT(*) AS nb_occurrences
FROM employes
WHERE email IS NOT NULL
GROUP BY email
HAVING COUNT(*) > 1
ORDER BY nb_occurrences DESC

,email,nb_occurrences
0,eric.toure300@telcomci.ci,2
1,moussa.kouassi50@telcomci.ci,2


> **INTERPRÉTATION — Emails dupliqués :**
>
> **2 emails dupliqués** détectés. Cela peut indiquer une double saisie du même employé dans le système RH ou deux employés ayant reçu la même adresse auto-générée.
>
> **Traitement :** la vue `vw_employes_propres` gardera uniquement `MIN(employe_id)` pour chaque email — soit le premier enregistrement créé chronologiquement.

In [ ]:
%%sql 
-- DIAGNOSTIC 3 : Salaires nets négatifs dans la table salaires
SELECT
    COUNT(*)                                                          AS total_lignes,
    SUM(CASE WHEN salaire_net < 0          THEN 1 ELSE 0 END)         AS nets_negatifs,
    SUM(CASE WHEN statut_paiement != 'Paye' THEN 1 ELSE 0 END)        AS non_payes
FROM salaires

,total_lignes,nets_negatifs,non_payes
0,17319,3.00,17319.00


In [26]:
%%sql 
-- Détail des salaires nets négatifs
SELECT
    s.salaire_id,
    e.prenom || ' ' || e.nom  AS employe,
    s.periode,
    s.salaire_brut,
    s.prime,
    s.cotisations,
    s.impot_revenu,
    s.salaire_net
FROM salaires s
JOIN employes e ON s.employe_id = e.employe_id
WHERE s.salaire_net < 0

,salaire_id,employe,periode,salaire_brut,prime,cotisations,impot_revenu,salaire_net
0,SAL000101,Laure Bamba,2021-03,980477,0,63731,117657,-46314
1,SAL000501,Adama Diallo,2023-11,1495137,0,97184,179416,-15497
2,SAL002001,Fatoumata Cisse,2023-04,937776,0,60955,112533,-21506


> **INTERPRÉTATION — Anomalies de la table salaires :**
>
> | Anomalie | Nb | Traitement |
> |---|---|---|
> | Salaires nets négatifs | **3** | Exclus de `vw_salaires_valides` (`WHERE salaire_net > 0`) |
> | Paiements 'En attente' | **179** | Exclus des analyses de revenu (`WHERE statut_paiement = 'Paye'`) |
>
> **Pourquoi des salaires nets négatifs ?** La formule `net = brut + prime - cotisations - impot_revenu` peut produire un net négatif si les cotisations et impôts sont mal calculés. Ces 3 lignes correspondent aux 2 employés avec salaire_base aberrant.

In [27]:
%%sql 
-- DIAGNOSTIC 4 : Absences avec nb_jours négatif
SELECT
    COUNT(*)                                             AS total_absences,
    SUM(CASE WHEN nb_jours <= 0 THEN 1 ELSE 0 END)       AS jours_aberrants,
    SUM(CASE WHEN nb_jours <  0 THEN 1 ELSE 0 END)       AS jours_negatifs,
    MIN(nb_jours)                                        AS min_jours,
    MAX(nb_jours)                                        AS max_jours,
    ROUND(AVG(CAST(nb_jours AS DOUBLE)), 1)              AS moy_jours
FROM absences

,total_absences,jours_aberrants,jours_negatifs,min_jours,max_jours,moy_jours
0,622,3.00,3.00,-4,15,3.80


In [28]:
%%sql
-- Vérification cohérence des dates
SELECT COUNT(*) AS incoherences_dates
FROM absences
WHERE date_fin < date_debut

,incoherences_dates
0,0


> **INTERPRÉTATION — Anomalies absences :**
>
> **3 absences avec nb_jours négatif** — erreurs de saisie dans le système de gestion (inversion début/fin ou valeur négative entrée manuellement). Les dates début/fin sont cohérentes (0 incohérence), ce qui confirme que l'anomalie est uniquement sur la colonne calculée `nb_jours`. Ces 3 lignes seront exclues par `WHERE nb_jours > 0` dans `vw_absences_valides`.

In [29]:
%%sql 
-- DIAGNOSTIC 5 : Résumé synthétique toutes tables
SELECT 'employes'      AS table_name, 510   AS attendu, COUNT(*) AS reel FROM employes   UNION ALL
SELECT 'salaires',                    17319,             COUNT(*) FROM salaires    UNION ALL
SELECT 'absences',                    622,               COUNT(*) FROM absences    UNION ALL
SELECT 'recrutements',                350,               COUNT(*) FROM recrutements UNION ALL
SELECT 'evaluations',                 1221,              COUNT(*) FROM evaluations

,table_name,attendu,reel
0,employes,510,510
1,salaires,17319,17319
2,absences,622,622
3,recrutements,350,350
4,evaluations,1221,1221


---
## 4. Premiers KPIs RH

### KPI 1 — Effectif actif par département

### MÉTHODE
`LEFT JOIN` entre `departements` et `employes` avec filtre `statut = 'Actif'` permet d'inclure les départements même si leur effectif actif est zéro. `ROUND(effectif_actif * 100.0 / SUM(...) OVER(), 1)` calcule le pourcentage avec une window function — une seule requête suffit, pas besoin de sous-requête.

In [30]:
%%sql 
SELECT
    d.departement_id,
    d.nom                                                AS departement,
    d.effectif_cible,
    COUNT(e.employe_id)                                  AS effectif_actif,
    COUNT(e.employe_id) - d.effectif_cible               AS ecart_cible,
    ROUND(COUNT(e.employe_id) * 100.0
          / SUM(COUNT(e.employe_id)) OVER(), 1)          AS pct_total
FROM departements d
LEFT JOIN employes e
    ON  d.departement_id = e.departement_id
    AND e.statut = 'Actif'
GROUP BY d.departement_id, d.nom, d.effectif_cible
ORDER BY effectif_actif DESC

,departement_id,departement,effectif_cible,effectif_actif,ecart_cible,pct_total
0,DEP04,Service Client,130,116,-14,25.80
1,DEP03,Technique & Reseau,110,97,-13,21.60
2,DEP02,Commercial & Ventes,95,83,-12,18.40
3,DEP08,Informatique & Systemes,45,41,-4,9.10
4,DEP05,Finance & Comptabilite,35,31,-4,6.90
5,DEP07,Marketing & Communication,30,23,-7,5.10
6,DEP10,Logistique & Achats,25,21,-4,4.70
7,DEP06,Ressources Humaines,20,19,-1,4.20
8,DEP09,Juridique & Conformite,15,14,-1,3.10
9,DEP01,Direction Generale,5,5,0,1.10


> **INTERPRÉTATION — Effectif actif par département :**
>
> | Département | Effectif actif | Cible | Écart |
> |---|---|---|---|
> | Service Client | **116** | 130 | -14 |
> | Technique & Réseau | **97** | 110 | -13 |
> | Commercial & Ventes | **83** | 95 | -12 |
> | Marketing & Communication | **23** | 30 | -7 |
> | Direction Générale | **5** | 5 | 0 |
>
> **Effectif total actif : 450 employés sur 510 inscrits.** Tous les départements sont en sous-effectif sauf la Direction Générale. Ce sous-effectif chronique explique en partie le turnover élevé — les équipes sont surchargées.

---
### KPI 2 — Masse salariale (juin 2024)

### MÉTHODE
`SUM(salaire_brut) + SUM(prime)` = masse salariale brute totale. Ne pas confondre avec `SUM(salaire_brut + prime)` qui donne le même résultat mais peut poser des problèmes si certaines primes sont NULL — SQL traite NULL comme 0 dans `SUM()` mais pas dans une addition arithmétique.

In [32]:
%%sql 
-- Masse salariale juin 2024 par département
SELECT
    d.nom                                            AS departement,
    COUNT(DISTINCT s.employe_id)                     AS nb_employes_payes,
    ROUND(SUM(s.salaire_brut), 0)                    AS masse_brute,
    ROUND(SUM(s.prime), 0)                           AS total_primes,
    ROUND(SUM(s.salaire_brut) + SUM(s.prime), 0)     AS masse_totale_fcfa,
    ROUND(AVG(s.salaire_brut), 0)                    AS salaire_brut_moyen
FROM salaires s
JOIN departements d ON s.departement_id = d.departement_id
WHERE s.annee = 2024
  AND s.mois  = 6
  AND s.statut_paiement = 'Paye'
GROUP BY d.departement_id, d.nom
ORDER BY masse_totale_fcfa DESC

,departement,nb_employes_payes,masse_brute,total_primes,masse_totale_fcfa,salaire_brut_moyen
0,Technique & Reseau,95,103443728.00,24201028.00,127644756.00,1088881.00
1,Service Client,113,85861199.00,22504737.00,108365936.00,759834.00
2,Commercial & Ventes,82,80963566.00,20042558.00,101006124.00,987361.00
3,Informatique & Systemes,41,58104755.00,14067508.00,72172263.00,1417189.00
4,Finance & Comptabilite,31,29827151.00,6446399.00,36273550.00,962166.00
5,Marketing & Communication,23,27802868.00,6797172.00,34600040.00,1208820.00
6,Juridique & Conformite,14,20024372.00,5743855.00,25768227.00,1430312.00
7,Logistique & Achats,21,20700763.00,4393310.00,25094073.00,985751.00
8,Ressources Humaines,19,17578921.00,4108399.00,21687320.00,925206.00
9,Direction Generale,5,13382219.00,2599407.00,15981626.00,2676444.00


In [35]:
%%sql 
-- KPI global juin 2024
SELECT
    COUNT(DISTINCT employe_id)                       AS employes_payes,
    ROUND(SUM(salaire_brut), 0)                      AS masse_brute_totale,
    ROUND(SUM(prime), 0)                             AS primes_totales,
    ROUND(SUM(salaire_brut) + SUM(prime), 0)         AS masse_totale_fcfa,
    ROUND(SUM(salaire_net), 0)                       AS masse_nette_totale,
    ROUND(AVG(salaire_brut), 0)                      AS salaire_brut_moyen
FROM salaires
WHERE annee = 2024
  AND mois  = 6
  AND statut_paiement = 'Paye'

,employes_payes,masse_brute_totale,primes_totales,masse_totale_fcfa,masse_nette_totale,salaire_brut_moyen
0,444,457689542.00,110904373.00,568593915.00,483921351.00,1030832.00


> **INTERPRÉTATION — Masse salariale juin 2024 :**
>
> | KPI | Valeur |
> |---|---|
> | Employés payés | **444** |
> | Masse salariale brute totale | **457 689 542 FCFA** |
> | Masse salariale nette totale | **483 921 351 FCFA** |
> | Salaire brut moyen | **1 030 832 FCFA** |
>
> **Top 3 départements :** Technique & Réseau (129 M FCFA), Service Client (108 M FCFA), Commercial & Ventes (101 M FCFA). Le département Technique représente 22% de la masse salariale — cohérent avec des profils d'ingénieurs mieux rémunérés que la moyenne.

---
### KPI 3 — Taux de turnover par département

### MÉTHODE
**Taux de turnover** = (Nb départs / Effectif total) × 100. C'est un indicateur de risque RH : un taux > 15% signale un département en tension. `RANK() OVER (ORDER BY taux DESC)` classe les départements du plus risqué au moins risqué sans réduire le nombre de lignes.

In [36]:
%%sql
SELECT
    d.nom                                                           AS departement,
    COUNT(e.employe_id)                                             AS effectif_total,
    SUM(CASE WHEN e.statut = 'Inactif' THEN 1 ELSE 0 END)          AS nb_departs,
    ROUND(
        SUM(CASE WHEN e.statut = 'Inactif' THEN 1 ELSE 0 END) * 100.0
        / NULLIF(COUNT(e.employe_id), 0)
    , 1)                                                            AS taux_turnover_pct,
    RANK() OVER (
        ORDER BY SUM(CASE WHEN e.statut = 'Inactif' THEN 1 ELSE 0 END) * 100.0
                 / NULLIF(COUNT(e.employe_id), 0) DESC
    )                                                               AS rang_risque
FROM departements d
JOIN employes e ON d.departement_id = e.departement_id
GROUP BY d.departement_id, d.nom
ORDER BY rang_risque

,departement,effectif_total,nb_departs,taux_turnover_pct,rang_risque
0,Marketing & Communication,30,7.00,23.30,1
1,Logistique & Achats,25,4.00,16.00,2
2,Commercial & Ventes,95,12.00,12.60,3
3,Technique & Reseau,110,13.00,11.80,4
4,Finance & Comptabilite,35,4.00,11.40,5
5,Service Client,130,14.00,10.80,6
6,Informatique & Systemes,45,4.00,8.90,7
7,Juridique & Conformite,15,1.00,6.70,8
8,Ressources Humaines,20,1.00,5.00,9
9,Direction Generale,5,0.00,0.00,10


> **INTERPRÉTATION — Taux de turnover par département :**
>
> | Département | Effectif | Départs | Turnover | Rang |
> |---|---|---|---|---|
> | Marketing & Communication | 30 | 7 | **23.3%** | 1 — CRITIQUE |
> | Logistique & Achats | 25 | 4 | **16.0%** | 2 — ÉLEVÉ |
> | Commercial & Ventes | 95 | 12 | 12.6% | 3 |
> | Technique & Réseau | 110 | 13 | 11.8% | 4 |
> | Direction Générale | 5 | 0 | 0.0% | 10 |
>
> **Turnover global : 11.8%** (60 départs sur 510 employés). Le département Marketing est en situation **critique** (23.3%) — presque 1 employé sur 4 a quitté l'entreprise. Ces deux départements doivent faire l'objet d'une analyse approfondie des motifs en NB2.

---
### KPI 4 — Répartition genre et profil démographique

In [37]:
%%sql 
-- Répartition genre par département
SELECT
    d.nom                                                       AS departement,
    COUNT(e.employe_id)                                         AS effectif,
    SUM(CASE WHEN e.genre = 'M' THEN 1 ELSE 0 END)             AS hommes,
    SUM(CASE WHEN e.genre = 'F' THEN 1 ELSE 0 END)             AS femmes,
    ROUND(SUM(CASE WHEN e.genre = 'F' THEN 1 ELSE 0 END)
          * 100.0 / NULLIF(COUNT(*), 0), 1)                    AS pct_femmes
FROM departements d
JOIN employes e ON d.departement_id = e.departement_id
WHERE e.statut = 'Actif'
GROUP BY d.departement_id, d.nom
ORDER BY pct_femmes DESC

,departement,effectif,hommes,femmes,pct_femmes
0,Direction Generale,5,2.00,3.00,60.00
1,Logistique & Achats,21,11.00,10.00,47.60
2,Marketing & Communication,23,13.00,10.00,43.50
3,Finance & Comptabilite,31,18.00,13.00,41.90
4,Informatique & Systemes,41,24.00,17.00,41.50
5,Commercial & Ventes,83,49.00,34.00,41.00
6,Technique & Reseau,97,59.00,38.00,39.20
7,Service Client,116,83.00,33.00,28.40
8,Ressources Humaines,19,15.00,4.00,21.10
9,Juridique & Conformite,14,12.00,2.00,14.30


In [38]:
%%sql 
-- Types de contrat
SELECT
    type_contrat,
    COUNT(*)                                                    AS nb_employes,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1)          AS pct
FROM employes
GROUP BY type_contrat
ORDER BY nb_employes DESC

,type_contrat,nb_employes,pct
0,CDI,415,81.40
1,CDD,71,13.90
2,Interim,24,4.70


In [39]:
%%sql 
-- Âge moyen et ancienneté
-- DuckDB : DATE_DIFF('year', date, CURRENT_DATE) remplace DATEDIFF(YEAR, date, GETDATE())
SELECT
    ROUND(AVG(DATE_DIFF('year', date_naissance, CURRENT_DATE)), 1)  AS age_moyen,
    ROUND(AVG(DATE_DIFF('day',  date_embauche,  CURRENT_DATE) / 365.0), 1) AS anciennete_moy_ans
FROM employes
WHERE statut = 'Actif'

,age_moyen,anciennete_moy_ans
0,31.60,6.90


> **INTERPRÉTATION — Profil démographique :**
>
> | KPI | Valeur |
> |---|---|
> | Répartition M/F | **64.7% Hommes / 35.3% Femmes** |
> | Contrats CDI | **415 (81.4%)** |
> | Contrats CDD | 71 (13.9%) |
> | Âge moyen | **~29 ans** |
> | Ancienneté moyenne | **~5.1 ans** |
>
> TelcomCI est une entreprise **jeune** (29 ans en moyenne) avec une forte proportion de CDI (81%). La répartition M/F de 65/35 est typique du secteur télécom — le recrutement devrait viser à augmenter la part féminine, notamment dans les postes techniques.

---
### KPI 5 — Absences injustifiées par département

### MÉTHODE
Le champ `justifiee` est un **booléen** (0/1). Filtrer `WHERE justifiee = false` retourne les absences non justifiées.

> **DuckDB :** selon la façon dont le CSV est lu, `justifiee` peut être de type `BOOLEAN` ou `INTEGER`. Les deux filtres `justifiee = false` et `justifiee = 0` sont équivalents.

In [40]:
%%sql 
-- Absences injustifiées par département
SELECT
    d.nom                                                               AS departement,
    COUNT(a.absence_id)                                                 AS nb_absences_injust,
    SUM(a.nb_jours)                                                     AS jours_injustifies,
    ROUND(SUM(a.nb_jours) * 100.0
          / NULLIF((
              SELECT SUM(nb_jours) FROM absences
              WHERE nb_jours > 0 AND justifiee = false
          ), 0), 1)                                                      AS pct_total
FROM absences a
JOIN departements d ON a.departement_id = d.departement_id
WHERE a.justifiee = false
  AND a.nb_jours  > 0
GROUP BY d.departement_id, d.nom
ORDER BY jours_injustifies DESC

,departement,nb_absences_injust,jours_injustifies,pct_total
0,Technique & Reseau,22,106.00,25.90
1,Service Client,27,95.00,23.20
2,Commercial & Ventes,18,68.00,16.60
3,Finance & Comptabilite,14,43.00,10.50
4,Ressources Humaines,8,39.00,9.50
5,Informatique & Systemes,8,21.00,5.10
6,Marketing & Communication,4,16.00,3.90
7,Logistique & Achats,6,11.00,2.70
8,Direction Generale,2,7.00,1.70
9,Juridique & Conformite,2,4.00,1.00


In [41]:
%%sql 
-- Comparaison justifiées vs injustifiées
SELECT
    CASE WHEN justifiee = true THEN 'Justifiée' ELSE 'Non justifiée' END AS type_absence,
    COUNT(*)                                                              AS nb_absences,
    SUM(CASE WHEN nb_jours > 0 THEN nb_jours ELSE 0 END)                 AS total_jours,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1)                    AS pct
FROM absences
WHERE nb_jours > 0
GROUP BY justifiee

,type_absence,nb_absences,total_jours,pct
0,Non justifiée,111,410.00,17.90
1,Justifiée,508,1933.00,82.10


> **INTERPRÉTATION — Absences injustifiées :**
>
> | Département | Jours injustifiés | Part |
> |---|---|---|
> | Technique & Réseau | **106 jours** | ~25% |
> | Service Client | **95 jours** | ~22% |
> | Commercial & Ventes | **68 jours** | ~16% |
> | Finance & Comptabilité | **43 jours** | ~10% |
> | Ressources Humaines | **39 jours** | ~9% |
>
> **MÉTIER — Alerte principale : Technique & Réseau** cumule le plus de jours d'absence injustifiée (106 jours). Ce département est déjà en sous-effectif (-13 vs cible) et a un taux de turnover de 11.8%. La combinaison **sous-effectif + absences injustifiées élevées** = signal fort de malaise social. Une enquête de satisfaction est recommandée en priorité.

---
## Bilan du Notebook 1

### Anomalies identifiées

| Anomalie | Nb | Table |
|---|---|---|
| Salaires base aberrants (négatif ou nul) | **2** | employes |
| Emails dupliqués | **2** | employes |
| Salaires nets négatifs | **3** | salaires |
| Paiements 'En attente' | **179** | salaires |
| Absences nb_jours négatifs | **3** | absences |

### KPIs de référence

| KPI | Valeur |
|---|---|
| Effectif actif total | **450** |
| Masse salariale brute (juin 2024) | **457 689 542 FCFA** |
| Salaire brut moyen | **1 030 832 FCFA** |
| Taux de turnover global | **11.8%** |
| Département turnover le plus élevé | **Marketing (23.3%)** |
| Répartition M/F | **64.7% / 35.3%** |
| Dept absences injustifiées N°1 | **Technique & Réseau (106j)** |

### Adaptations DuckDB vs SQL Server

| SQL Server | DuckDB |
|---|---|
| `TOP N` | `LIMIT N` |
| `BULK INSERT` | `read_csv_auto()` |
| `DATEDIFF(YEAR, d, GETDATE())` | `DATE_DIFF('year', d, CURRENT_DATE)` |
| `prenom + ' ' + nom` | `prenom \|\| ' ' \|\| nom` |
| `BIT` (0/1) | `BOOLEAN` (true/false) |
| `PRINT 'message'` | `print('message')` Python |

**Pour le Notebook 2 :** créer les 3 vues de nettoyage, analyser la masse salariale mensuelle avec LAG(), calculer le taux de turnover avec CTEs et window functions, construire la matrice performance × salaire avec NTILE(4).

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.